<a href="https://colab.research.google.com/github/moiseevaaleksandra5757-stack/python-ai-Moiseeva-Sasha/blob/main/notebooks/week2b_read_csv.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📊 Week 2: Data Analysis — Чтение и проверка данных

Цель: Научиться читать CSV-файлы из репозитория GitHub в Google Colab и выполнять базовую проверку данных с помощью pandas.

Данные:
- **[`music_composer_genre.csv`](https://github.com/moiseevaaleksandra5757-stack/python-ai-Moiseeva-Sasha/blob/main/data/music_composer_genre.csv) — произведения, авторы, жанры, страны и продолжительность музыкальных композиций**

Что мы делаем:
1. Клонируем репозиторий GitHub в Colab
2. Читаем CSV-файлы в pandas DataFrame
3. Очищаем и переименовываем столбцы
4. Смотрим структуру данных и делаем быструю валидацию

## 🐱 [1] Клонируем репозиторий курса в Colab

In [3]:
# 🐱 Шаг 1. Клонируем репозиторий курса в Colab

import os

# **ИЗМЕНЕНО: имя репозитория и URL**
repo = "python-ai-Moiseeva-Sasha"  # было: "python-ai-template"
repo_path = f"/content/{repo}"

if not os.path.exists(repo_path):
    # **ИЗМЕНЕНО: URL репозитория**
    !git clone -q https://github.com/moiseevaaleksandra5757-stack/python-ai-Moiseeva-Sasha.git

if os.getcwd() != repo_path:
    %cd {repo_path}

print("✅ Репозиторий готов, теперь мы работаем внутри папки", repo)

✅ Репозиторий готов, теперь мы работаем внутри папки python-ai-Moiseeva-Sasha


## 📥 [2A] Простое чтение CSV-файлов в pandas

Сначала просто прочитаем оба CSV-файла в объекты `DataFrame`, без каких‑либо изменений.

После этого мы узнаем, сколько строк загружено в каждый датасет.

In [4]:
# 🐱 Шаг 2A. Чтение CSV-файлов в pandas

import pandas as pd

# **ИЗМЕНЕНО: путь к файлу и имя переменной**
df_music = pd.read_csv("data/music_composer_genre.csv")  # было: "data/examples/cartoons_genre_country_duration.csv"

# **ИЗМЕНЕНО: убран второй DataFrame, т.к. у вас один файл**
# df_assessment = pd.read_csv("data/examples/cartoons_assessment_reviews.csv")

print("✅ Загружено строк в df_music:", len(df_music))  # было: df_genre
# print("✅ Загружено строк в df_assessment:", len(df_assessment))  # удалено

✅ Загружено строк в df_music: 902


## 🧹 [2B] Очистка и переименование столбцов

В вашем CSV-файле `music_composer_genre.csv` есть **технические столбцы**, которые полезны для Викиданных, но мешают простому анализу:

- Столбец `work` с URL (ссылкой на объект Wikidata) — **удаляем**, так как в базовом анализе он не требуется.
- Столбцы `workLabel`, `authorLabel`, `genreLabel`, `countryLabel`, `durationLabel` содержат читаемые подписи (названия).

В этом шаге мы:
- **удаляем** столбец с URL Wikidata (`work`);
- переименовываем столбцы, убирая постфикс `Label`:
  - `workLabel` → `work`
  - `authorLabel` → `author`
  - `genreLabel` → `genre`
  - `countryLabel` → `country`
  - `durationLabel` → `duration`
- приводим столбец `duration` к числовому типу, заменяя пропуски на 0.

🔁 **Идемпотентность**: код проверяет текущую структуру данных и пропускает шаги, если они уже выполнены. Это значит, что ячейку можно запускать многократно без ошибок и без порчи данных.

При приведении к числам мы используем:
- `pd.to_numeric(..., errors="coerce")` — преобразует значения в числа, некорректные значения превращает в `NaN`;
- `fillna(0)` — заменяет пропущенные значения (`NaN`) на 0;
- `astype(int)` — переводит столбец к целочисленному типу.

In [9]:
# 🧹 Шаг 2B. Очистка и переименование столбцов для df_music
# 🔁 Идемпотентно: можно запускать многократно без ошибок

def clean_music_dataframe(df):
    """
    Очищает и переименовывает столбцы в DataFrame.
    Работает корректно при повторных запусках.
    """
    # === Проверка: данные уже очищены? ===
    expected_clean_cols = {"work", "author", "genre", "country", "duration"}
    current_cols = set(df.columns)

    # Если все ожидаемые "чистые" колонки уже есть — пропускаем
    if expected_clean_cols.issubset(current_cols):
        # Дополнительно убедимся, что "грязных" колонок точно нет
        label_cols_present = [c for c in current_cols if c.endswith("Label")]
        if not label_cols_present and "work" in current_cols and df["work"].dtype == "object" and len(df) > 0:
            # Проверяем, что work — это не URL, а уже название произведения
            sample = str(df["work"].iloc[0]).strip()
            if not sample.startswith("http"):
                print("⏭️ df_music уже очищен, пропускаем преобразования")
                return df

    # === Выполняем очистку ===
    df_clean = df.copy()  # работаем с копией, чтобы не ломать оригинал при ошибке

    # 1) Удаляем столбец с URL, если он ещё есть (начинается с http)
    if "work" in df_clean.columns:
        # Проверяем, содержит ли столбец URL-адреса
        sample_val = str(df_clean["work"].iloc[0]).strip() if len(df_clean) > 0 else ""
        if sample_val.startswith("http://") or sample_val.startswith("https://"):
            df_clean = df_clean.drop(columns=["work"])
            print("🗑️ Удалён столбец 'work' с URL Wikidata")

    # 2) Переименовываем столбцы *Label → без постфикса (если они ещё есть)
    rename_map = {
        "workLabel": "work",
        "authorLabel": "author",
        "genreLabel": "genre",
        "countryLabel": "country",
        "durationLabel": "duration",
    }
    # Переименовываем только те колонки, которые реально существуют
    existing_renames = {k: v for k, v in rename_map.items() if k in df_clean.columns}
    if existing_renames:
        df_clean = df_clean.rename(columns=existing_renames)
        print(f"✏️ Переименованы столбцы: {list(existing_renames.keys())}")

    # 3) Приводим duration к числовому типу (только если столбец есть и не числовой)
    if "duration" in df_clean.columns and not pd.api.types.is_numeric_dtype(df_clean["duration"]):
        df_clean["duration"] = pd.to_numeric(
            df_clean["duration"], errors="coerce"
        ).fillna(0).astype(int)
        print("🔢 Столбец 'duration' преобразован в int")
    elif "duration" in df_clean.columns:
        print("✅ Столбец 'duration' уже числовой")

    return df_clean

# === Применяем функцию ===
df_music = clean_music_dataframe(df_music)

print("\n✅ Данные готовы к анализу")
print(f"📋 Итоговые колонки: {df_music.columns.tolist()}")

⏭️ df_music уже очищен, пропускаем преобразования

✅ Данные готовы к анализу
📋 Итоговые колонки: ['work', 'author', 'genre', 'country', 'duration']


## 🔍 [3] Обзор данных: структура и первые строки

Сделаем короткий обзор обоих DataFrame:

- посмотрим размер таблицы (`shape`);
- выведем список столбцов;
- посмотрим первые несколько строк;
- дополнительно посчитаем базовую статистику по бюджету (`capital_cost`).

Для удобства напишем маленькую функцию `show_info(df, name)`, чтобы не повторять один и тот же код два раза.

In [ ]:
def show_info(df, name, n=5):
    """Краткий обзор DataFrame: имя, размер, список столбцов и первые строки."""
    print(f"\n📊 {name}")
    print("Размер:", df.shape)
    print("Столбцы:", ", ".join(df.columns))
    print("\nПервые строки:")
    print(df.head(n))

# 🔍 Шаг 3. Обзор данных

show_info(df_genre, "Жанры, страны и продолжительность (df_genre)")
show_info(df_assessment, "Год выпуска, оценки и рейтинги (df_assessment)")


📊 Жанры, страны и продолжительность (df_genre)
Размер: (2596, 6)
Столбцы: URL, film, genre, country, duration, capital_cost

Первые строки:
                                       URL                       film  \
0  http://www.wikidata.org/entity/Q1128756                Мэри и Макс   
1  http://www.wikidata.org/entity/Q1199692  Отважный маленький тостер   
2  http://www.wikidata.org/entity/Q1514402                  Ренессанс   
3  http://www.wikidata.org/entity/Q1514402                  Ренессанс   
4  http://www.wikidata.org/entity/Q1418615     Приключения кота Фрица   

                                    genre     country  duration  capital_cost  
0                       взрослая анимация   Австралия        90     8240000.0  
1  экранизация литературного произведения         США        90     2300000.0  
2                               киберпанк  Люксембург       101    18000000.0  
3                                 неонуар     Франция       101    18000000.0  
4  экранизация литер

## ✅ [4] Быстрая проверка и валидация данных

Здесь мы посмотрим:

- сколько **уникальных** фильмов, стран, жанров есть в данных;
- **какие страны встречаются чаще всего** (Топ‑5 по числу строк);
- **какие жанры самые популярные** (Топ‑10 по числу строк);
- **какие оценки (assessment) и результаты (outcome) присутствуют** в данных об оценках.

Функция `value_counts()`:
- считает, сколько раз каждое значение встречается в столбце;
- сортирует результаты по убыванию.

Метод `.head()` берёт первые N строк, поэтому
`df_genre["country"].value_counts().head()` даёт **Топ‑5 стран по числу записей**.

In [ ]:
# ✅ Шаг 4. Быстрая проверка и валидация данных

print("🔍 Быстрая проверка данных")

# Датасет 1: жанры, страны, длительность
print("\n📊 Датасет: Жанры, страны и продолжительность")
print("Уникальных мультфильмов в df_genre:", df_genre["film"].nunique())
print("Уникальных стран:", df_genre["country"].nunique())
print("Уникальных жанров:", df_genre["genre"].nunique())

print("\nТоп-5 стран по числу записей:")
print(df_genre["country"].value_counts().head())

print("\nТоп-10 жанров:")
print(df_genre["genre"].value_counts().head(10))

# Датасет 2: оценки и рейтинги
print("\n📊 Датасет: Год выпуска, оценки и рейтинги")
print("Уникальных мультфильмов в df_assessment:", df_assessment["film"].nunique())
print("Уникальных типов оценок:", df_assessment["assessment"].nunique())
print("Диапазон лет:", df_assessment["publicationYear"].min(), "—", df_assessment["publicationYear"].max())

print("\nТипы оценок (assessment):")
print(df_assessment["assessment"].value_counts())

print("\nРезультаты оценок (outcome):")
print(df_assessment["outcome"].value_counts())

print("\nСтатистика по годам публикации:")
print(df_assessment["publicationYear"].describe())

🔍 Быстрая проверка данных

📊 Датасет: Жанры, страны и продолжительность
Уникальных мультфильмов в df_genre: 418
Уникальных стран: 40
Уникальных жанров: 125

Топ-5 стран по числу записей:
country
США               1303
Франция            196
Великобритания     125
Россия              91
Дания               82
Name: count, dtype: int64

Топ-10 жанров:
genre
приключенческий фильм          353
комедийный фильм               305
фэнтезийный фильм              289
семейный фильм                 247
детский фильм                  187
музыкальный фильм              120
драматический фильм             91
боевик                          84
научно-фантастический фильм     74
мультфильм                      70
Name: count, dtype: int64

📊 Датасет: Год выпуска, оценки и рейтинги
Уникальных мультфильмов в df_assessment: 4929
Уникальных типов оценок: 22
Диапазон лет: 0 — 2027

Типы оценок (assessment):
assessment
тест Бекдел                                                                   1426
обрат

## 📝 Summary

**Что мы сделали в этом ноутбуке (Week 2):**

- ✅ Клонировали репозиторий GitHub в Colab
- ✅ Прочитали 2 CSV-файла из `data/examples/`
- ✅ Удалили URL Wikidata и переименовали столбцы (`*Label → короткие имена`)
- ✅ Проверили структуру данных (размер, столбцы, первые строки)
- ✅ Выполнили быструю валидацию:
  - количество уникальных фильмов, стран, жанров
  - диапазоны значений
  - топ стран и жанров по числу записей
  - типы оценок и результатов

Теперь у нас есть **аккуратные, проверенные таблицы**, с которыми удобно работать дальше.

В отдельном ноутбуке для следующей недели мы будем использовать **те же данные** для:
- более сложного анализа (группировки, фильтрация),
- и построения визуализаций (графики и диаграммы). 🎨